In [ ]:
!pip install newspaper3k

In [ ]:
!pip install selenium

In [ ]:
!pip install webdriver_manager

In [21]:
!pip install summa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 618.4 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for summa: filename=summa-1.2.0-py3-none-any.whl size=54389 sha256=8092ef7e04ee34ec1b619ade874bb24e80eafdd04c3995a64cdc526c22cddf37
  Stored in directory: /root/.cache/pip/wheels/4a/ca/c5/4958614cfba88ed6ceb7cb5a849f9f89f9ac49971616bc919f
Successfully built summa


In [18]:
from newspaper import Article
from bs4 import BeautifulSoup
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
browser = webdriver.Chrome(options=options)

# 카테고리 별 100개 기사 크롤링 (링크)
# 한 페이지에 20개

url_list = []   # url 저장 리스트
a_list = []     # a 태그 저장 리스트

for category in range(1):     # 6
    for page in range(1, 2):  # 1, 6
        url = f'https://news.naver.com/main/main.naver?mode=LSD&mid=shm&sid1={100 + category}#&date=%2000:00:00&page={page}'
        browser.get(url)

        time.sleep(1)

        soup = BeautifulSoup(browser.page_source, "html.parser")
        a_list.extend(soup.select(".type06_headline dt+dt a"))
        a_list.extend(soup.select(".type06 dt+dt a"))

        print("카테고리 : ", 100 + category, "페이지 : ", page)

for a in a_list:
    url_list.append(a["href"])  # 링크

# 수집한 링크의 제목, 본문 크롤링
# + 날짜 추가하기

title_list = []
content_list = []
title = []
content = []

for url in url_list:

      browser.get(url)

      time.sleep(1)

      soup = BeautifulSoup(browser.page_source, "html.parser")

      title.extend(soup.select("#title_area span"))
      content.extend(soup.select("#dic_area"))

for t in title:
  title_list.append(t.text)

for c in content:
  c = c.text.replace('\n', '')
  c = c.replace('\xa0', '')
  c = c.replace('\t', '')
  content_list.append(c)

카테고리 :  100 페이지 :  1


In [19]:
article_df = pd.DataFrame({"제목" : title_list,
                          "본문" : content_list,
                           "링크" : url_list})

print(article_df)

                                                제목  \
0             엄숙한 국감장, 질의 끝나자 국회의원들 사인받았다…증인 누구길래?   
1           국감 가장 돋보인 의원 용혜인, 이탄희, 한준호 [한국갤럽 여론조사]   
2                        김승겸 합참 “작전 둔갑시켰다는건 명예 손상”   
3                    양평 고속도로 특혜 의혹이 '타진요'라는 원희룡 장관   
4           軍, 북한 목선 '경계 실패' 논란에 "인정할 수 없다… 대응 적절"   
5                     ICBM 킬러 '존 핀' 제주 입항 [뉴시스Pic]   
6    "박근혜 前 대통령 탄핵 때도...윤석열 대통령 지지율, 정말 최악" [Y녹취록]   
7                [정치쇼] 전청조와 직접 통화해보니…"저 전청조 아니라고요"   
8                  ‘역도 영웅’에서 ‘최연소 차관’…장미란 재산이 공개됐다   
9       "적 장사정포 위협 완전 궤멸" 지구사 대화력전 FTX 훈련 [뉴시스Pic]   
10   감사원 핵심 '타이거' 비결은 무협지?...유병호 총장의 지휘 노트 [앵커리포트]   
11            [뉴스초점] '인요한 혁신위' 첫 회의…민주 지명직 최고위원 인선   
12       "절도범이 들여온 고려불상, 일본 소유권" 대법 판결에, 日 조기반환 촉구   
13                 'TK의원 비만 고양이' 비난한 이준석 대구 출마설 솔솔   
14          北 억압 멈추라는 野에, 신원식 “강도 아닌 경찰 나쁘다는 본말전도”   
15  "TK 유권자에게 메시지 던져"vs"대통령 행보 균형감 없어"[박영환의 시사1번지]   
16    장미란 문체 2차관 재산 6.9억 신고…정기석 건보공단 이사장 91억 최고 재산   
17           여야 '양평고속도로' '채상

In [23]:
from summa.summarizer import summarize  # 이 라이브러리는 장문일수록 정확도가 높지만 전반으로 상태 이상함 + 짧은 기사는 요약 시도도 안함

In [ ]:
!pip install textrankr

In [ ]:
!pip install krwordrank

In [64]:
from textrankr import TextRank  # 문장 요약 라이브러리
from krwordrank.word import KRWordRank  # 키워드 추출 라이브러리

# 본문을 리스트 형식으로 저장하기 위한 클래스
class MyTokenizer:
  def __call__(self, text: str) -> list[str]:
    tokens: list[str] = text.split()
    return tokens

# 요약 하는 공간 생성
mytokenizer: MyTokenizer = MyTokenizer()
textrank : TextRank = TextRank(mytokenizer)

# 제목 키워드 추출
# wordrank_extractor = KRWordRank(5, 10)


for i in range(3): # 임시로 3개만 
  title, content, url = article_df['제목'].values[i], article_df['본문'].values[i], article_df['링크'].values[i]

# 키워드 추출 파트 -> 일단 보류(동작 에러)
# for word, r in sorted(keywords.items(), key=lambda x: x[1], reverse=True)[:30]:
#   print('%8s:\t%.4f' % (word, r))

  sumContent: str = textrank.summarize(content, 5)  # 숫자는 요약문 줄 수 설정

  print(sumContent, '\n\n')

# 키워드 추출 : https://soyoung-new-challenge.tistory.com/45

26일 서울 여의도 국회에서 열린 국회 산업통상자원중소벤처기업위원회의 국정감사가 정회되자 증인으로 출석한 가수 겸 배우 김민종 KC컨텐츠 공동대표에게 참석자들이 말을 걸고 있다
증인으로 출석한 가수 겸 배우 김민종 KC컨텐츠 공동대표였다
김민종 KC컨텐츠 대표가 26일 오후 서울 여의도 국회에서 열린 산업통상자원중소벤처기업위원회의 산업통상자원부에 대한 종합감사에서 증인 선서를 하고 있다
그는 “저는 데뷔한 지 35년 된 배우”라며 “오늘 이후로 제가 무슨 사업가로 전환된 것 같다”고 했다
의원들은 김씨에게 다가가 사인을 요청하기도 했다 



임기 막바지인 21대 국회에 대한 부정 평가가 긍정 평가보다 압도적으로 높게 나타났다
‘잘못했다’가 80%에 달한 반면 ‘잘했다’는 13%에 그쳤다
7%는 의견을 보류했다
21대 국회, ‘잘못했다’ 80%21대 국회의 임기 시작 직전인 2020년 5월의 조사에서는 유권자 중 63%가 ‘잘할 것’이라고 내다봤는데, “국회(정치권)가 그런 바람을 충족하지 못한 것으로 보인다”라고 한국갤럽은 지적했다
한국갤럽다만 19대와 20대 국회 역시 임기 막바지엔 21대와 비슷한 평가를 받았다 



야당 의원들은 이날 국감에서 민간의 신고 이후 군이 현장에 전력을 보냈고, 북한 목선이 NLL을 넘어오는 것을 포착하지 못했다는 이유로 '실패한 작전'이라고 지적하자 김 의장이 반박한 것이다
합참은 이날 언론에 배포한 '북한 소형 목선 관련 경과 및 조치'라는 문서를 통해 지난 24일 새벽 3시부터 동해 NLL 이북 해역에서 북한 특이징후가 포착돼 구축함을 보냈고, 북한 단속선에 대응했다고 밝혔다
북한 단속선은 동해 NLL을 넘어 남하하는 북한 목선을 추격하고 있던 것으로 군 당국은 추정했다
군이 현장 출동을 준비하던 7시 10분 민간 선박이 북한 목선을 발견해 해경에 신고했고, 해경으로부터 신고 내용을 통보받은 군 당국은 추적 중이던 선박과 같은 선박임을 확인했다
군 당국은 7시 15분부터 해상초계기와 함정을 북한 목선이 있는 현장에 보냈